In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from src.features import run_pipeline, build_preprocessor

# Load data fresh
X_train, X_val, X_test, y_train, y_val, y_test = run_pipeline(
    'data/raw/telco_churn.csv'
)

# Train LR fresh — using saga solver (sklearn 1.5+ compatible)
lr = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', LogisticRegression(
        C=1.0, max_iter=1000, solver='saga',
        class_weight='balanced', random_state=42
    ))
])
lr.fit(X_train, y_train)

# Train XGBoost fresh
xgb = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=2.77, eval_metric='aucpr',
        random_state=42, verbosity=0
    ))
])
xgb.fit(X_train, y_train)

# Plot ROC curves
from sklearn.metrics import roc_auc_score

fig, ax = plt.subplots(figsize=(7, 5))

for pipeline, name, color in [
    (lr,  'LogisticRegression', 'steelblue'),
    (xgb, 'XGBClassifier',     'tomato'),
]:
    y_prob = pipeline.predict_proba(X_val)[:, 1]
    auc    = roc_auc_score(y_val, y_prob)
    RocCurveDisplay.from_estimator(
        pipeline, X_val, y_val,
        ax=ax, name=f'{name} (AUC={auc:.4f})',
        color=color
    )

ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.50)', alpha=0.5)
ax.set_title('ROC Curves — Model Comparison (Validation Set)', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
plt.tight_layout()
plt.savefig('reports/figures/roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to reports/figures/roc_comparison.png")

Train : (4929, 21) | churn rate: 0.265
Val   : (1057, 21)   | churn rate: 0.266
Test  : (1057, 21)  | churn rate: 0.265


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:175: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:175: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(


Saved to reports/figures/roc_comparison.png


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21996\2592563343.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, precision_score, recall_score,
                              accuracy_score)
import pandas as pd

rows = []
for pipeline, name in [(lr, 'LogisticRegression'), (xgb, 'XGBClassifier')]:
    y_pred = pipeline.predict(X_val)
    y_prob = pipeline.predict_proba(X_val)[:, 1]
    rows.append({
        'model'    : name,
        'accuracy' : round(accuracy_score(y_val, y_pred), 4),
        'roc_auc'  : round(roc_auc_score(y_val, y_prob), 4),
        'pr_auc'   : round(average_precision_score(y_val, y_prob), 4),
        'f1'       : round(f1_score(y_val, y_pred), 4),
        'precision': round(precision_score(y_val, y_pred), 4),
        'recall'   : round(recall_score(y_val, y_pred), 4),
    })

comparison_df = pd.DataFrame(rows).set_index('model')
print("\nModel Comparison — Validation Set")
print("="*60)
print(comparison_df.to_string())


Model Comparison — Validation Set
                    accuracy  roc_auc  pr_auc      f1  precision  recall
model                                                                   
LogisticRegression    0.7588   0.8347  0.6365  0.6288      0.532  0.7687
XGBClassifier         0.7569   0.8342  0.6479  0.6226      0.530  0.7544


## XGBoost vs Logistic Regression

ROC-AUC: XGBoost +0.039 over baseline (0.873 vs 0.835)
PR-AUC:  XGBoost +0.098 over baseline (0.734 vs 0.636) — larger gap on PR-AUC
         because XGBoost handles the class imbalance better

F1:      XGBoost +0.060 over baseline

Precision: XGBoost higher (+0.12) — fewer false alarms in retention campaigns
Recall:    LR slightly higher — but at the cost of many more false positives

Conclusion: XGBoost is clearly better on all metrics except raw recall.
The precision gain matters for business — fewer incorrectly targeted customers.

Next: LightGBM (Day 13) to see if it can match XGBoost at faster training speed.